In [12]:
import fitz
import json
import asyncio
import time
from collections import Counter
import ollama
import nest_asyncio

# Necessary for running in environments like Jupyter or Spyder
nest_asyncio.apply()

class BatchPDFKeywordPipeline:
    def __init__(self, model_name="qwen2.5:7b", batch_size=3, max_concurrency=2, rate_limit=2):
        self.model_name = model_name
        self.batch_size = batch_size
        self.steps = []
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.rate_limit = rate_limit
        self.last_call_time = 0

    # -----------------------------
    # PIPELINE CONTROL
    # -----------------------------
    def add_step(self, func):
        self.steps.append(func)

    async def run(self, input_data):
        data = input_data
        for step in self.steps:
            print(f"Running: {step.__name__}")
            if asyncio.iscoroutinefunction(step):
                data = await step(self, data)
            else:
                data = step(self, data)
        return data

    async def rate_limiter(self):
        min_interval = 1.0 / self.rate_limit
        elapsed = time.time() - self.last_call_time
        if elapsed < min_interval:
            await asyncio.sleep(min_interval - elapsed)
        self.last_call_time = time.time()

    # -----------------------------
    # IMPROVED PROMPT
    # -----------------------------
    def build_batch_prompt(self, batch):
        # We include the ID, Topic Name, and Text to give the LLM maximum context for keywords
        return f"""
You are a Retail Banking Domain Expert on Loans. 

TASK:
Assign EXACTLY 5 high-quality banking keywords to each ID provided below.

INSTRUCTIONS:
1. Use the 'text' provided as a primary reference.
2. If the text is short or empty, use your OWN domain knowledge based on the 'topic_name' and 'section_name' to assign relevant keywords.
3. Every single ID MUST have exactly 5 strings. 
4. NEVER return "N/A", "None", or empty lists.
5. Use multi-word professional phrases (e.g., "Customer Due Diligence", "Amortization Schedule", "Credit Risk Assessment").

OUTPUT FORMAT:
Return ONLY valid JSON. 
{{
  "ID": ["Keyword 1", "Keyword 2", "Keyword 3", "Keyword 4", "Keyword 5"]
}}

INPUT DATA:
{json.dumps(batch, indent=2)}
"""

    # -----------------------------
    # LLM BATCH PROCESSING
    # -----------------------------
    async def process_batch(self, batch):
        async with self.semaphore:
            await self.rate_limiter()
            try:
                response = await asyncio.to_thread(
                    ollama.chat,
                    model=self.model_name,
                    format="json",
                    messages=[
                        {"role": "system", "content": "You are a JSON generator. Every key must have 5 strings. Never return empty arrays or nulls."},
                        {"role": "user", "content": self.build_batch_prompt(batch)}
                    ],
                    options={"temperature": 0.2} # Small temperature helps the model "choose" when it's unsure
                )
                
                raw = response["message"]["content"].strip()
                data = json.loads(raw)
                return data if isinstance(data, dict) else {}

            except Exception as e:
                print(f"Batch processing error: {e}")
                return {}

    # -----------------------------
    # STEP 1: EXTRACT SPANS
    # -----------------------------
    def step_extract_spans(self, pdf_path):
        doc = fitz.open(pdf_path)
        spans = []
        for page_num, page in enumerate(doc):
            for block in page.get_text("dict")["blocks"]:
                if "lines" not in block: continue
                for line in block["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        if text:
                            spans.append({
                                "text": text, 
                                "size": round(span["size"], 1)
                            })
        return spans

    # -----------------------------
    # STEP 2: PARSE HIERARCHY
    # -----------------------------
    def step_parse_hierarchy(self, spans):
        sizes = [s["size"] for s in spans]
        freq = Counter(sizes)
        sorted_sizes = sorted(freq.keys(), reverse=True)
        
        # Section = Largest font, Topic = Second largest
        section_font = sorted_sizes[0]
        topic_font = sorted_sizes[1]

        results = []
        current_section, current_topic = None, None
        sc, tc = 0, 0

        for span in spans:
            text, size = span["text"], span["size"]
            
            if size == section_font:
                sc += 1; tc = 0
                current_section = {"section_id": f"S{sc:03d}", "section_name": text, "topics": []}
                results.append(current_section)
                current_topic = None
            
            elif size == topic_font and current_section:
                tc += 1
                current_topic = {
                    "topic_id": f"{current_section['section_id']}_T{tc:03d}",
                    "topic_name": text,
                    "lines": []
                }
                current_section["topics"].append(current_topic)
            
            elif current_topic:
                current_topic["lines"].append(text)
        
        return results

    # -----------------------------
    # STEP 3: SEQUENTIAL MERGE
    # -----------------------------
    def step_merge_topics_sequentially(self, data):
        """Groups and merges all text into a single block per topic name."""
        flattened_topics = []
        for section in data:
            for topic in section["topics"]:
                # Merge lines and clean up whitespace
                raw_text = " ".join(topic["lines"]).strip()
                clean_text = " ".join(raw_text.split()) 

                flattened_topics.append({
                    "topic_id": topic["topic_id"],
                    "topic_name": topic["topic_name"],
                    "section_id": section["section_id"],
                    "section_name": section["section_name"],
                    "text": clean_text
                })
        return flattened_topics

    # -----------------------------
    # STEP 4: ENRICH WITH KEYWORDS
    # -----------------------------
    async def step_enrich_topic_keywords(self, flattened_topics):
        batches = [
            flattened_topics[i:i+self.batch_size]
            for i in range(0, len(flattened_topics), self.batch_size)
        ]

        tasks = []
        for batch in batches:
            # We pass IDs, Topic Names, and Section Names for fallback context
            payload = [
                {
                    "id": t["topic_id"], 
                    "topic_name": t["topic_name"], 
                    "section_name": t["section_name"],
                    "text": t["text"][:2000] # Cap text length for model context limits
                } 
                for t in batch
            ]
            tasks.append(asyncio.create_task(self.process_batch(payload)))

        all_keywords = {}
        batch_results = await asyncio.gather(*tasks)
        for res in batch_results:
            all_keywords.update(res)

        # Map results back to the topics
        for topic in flattened_topics:
            tid = topic["topic_id"]
            topic["keywords"] = all_keywords.get(tid, [])
            
            # Final sanity check: if the LLM still returned less than 5
            while len(topic["keywords"]) < 5:
                topic["keywords"].append("N/A")

        return flattened_topics


# -----------------------------
# EXECUTION
# -----------------------------
async def main():
    # Adjusted batch_size for better stability
    pipeline = BatchPDFKeywordPipeline(batch_size=3, max_concurrency=2)

    pipeline.add_step(BatchPDFKeywordPipeline.step_extract_spans)
    pipeline.add_step(BatchPDFKeywordPipeline.step_parse_hierarchy)
    pipeline.add_step(BatchPDFKeywordPipeline.step_merge_topics_sequentially)
    pipeline.add_step(BatchPDFKeywordPipeline.step_enrich_topic_keywords)

    pdf_path = r"C:\Users\prasa\Downloads\loan_products_final.pdf"
    
    try:
        result = await pipeline.run(pdf_path)

        with open("final_merged_keywords.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=4, ensure_ascii=False)

        print(f"Success! Processed {len(result)} unique topic sections.")
    except Exception as e:
        print(f"Pipeline failed: {e}")

if __name__ == "__main__":
    asyncio.run(main())

Running: step_extract_spans
Running: step_parse_hierarchy
Running: step_merge_topics_sequentially
Running: step_enrich_topic_keywords
Success! Processed 138 unique topic sections.


In [13]:
import uuid
from typing import List, Dict, Any, Optional
from sqlmodel import SQLModel, Field
from sqlalchemy import Column
from sqlalchemy.dialects.postgresql import JSONB
from pgvector.sqlalchemy import Vector


class ChunkModel(SQLModel, table=True):
    __tablename__ = "document_chunks"

    # 🔹 Primary key
    id: uuid.UUID = Field(default_factory=uuid.uuid4, primary_key=True)

    # 🔹 Core content
    chunk_text: str = Field(nullable=False)

    # 🔹 Embedding (768 for nomic)
    embedding: Optional[List[float]] = Field(
        default=None,
        sa_column=Column(Vector(768))
    )

    # 🔹 Metadata JSON
    chunk_metadata: Dict[str, Any] = Field(
        default_factory=dict,
        sa_column=Column(JSONB)
    )

    # 🔹 Chunk linking
    prev_chunk_id: Optional[uuid.UUID] = Field(default=None, foreign_key="document_chunks.id")
    next_chunk_id: Optional[uuid.UUID] = Field(default=None, foreign_key="document_chunks.id")


c:\Users\prasa\anaconda3\envs\olm-env\lib\site-packages\sqlalchemy\sql\crud.py:34: RuntimeWarning: coroutine 'main' was never awaited
  from . import dml
